# Taller cap. 6 – Punto 1: multiplicación de matrices 2×2 con Machine Learning

**Idea (figura 6.1):** en programación tradicional damos *datos + reglas* y obtenemos *respuestas*. En ML damos *datos + respuestas* y el algoritmo encuentra las *reglas (modelo)*.

Aquí los datos son pares de matrices A y B, las respuestas son C = A·B, y el modelo debe aprender a multiplicar.

**Pasos:** 1) generar el data set, 2) partirlo en entrenamiento/validación/prueba, 3) entrenar modelos, 4) probar con 10 ejemplos vs. cálculo analítico, 5) comparar costo computacional.

In [ ]:
import time
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error

rng = np.random.default_rng(42)   # semilla fija para que los resultados se repitan

## 1. Data set de 50.000 multiplicaciones
- `A` y `B`: enteros aleatorios entre -20 y 20 (`integers(-20, 21)` excluye el 21).
- `C = A @ B` son las respuestas correctas (los *labels*).
- Cada matriz 2×2 se aplana a 4 números, así que cada ejemplo tiene **8 entradas** (a11,a12,a21,a22,b11,...,b22) y **4 salidas** (c11,c12,c21,c22).

In [ ]:
N = 50_000
A = rng.integers(-20, 21, size=(N, 2, 2))
B = rng.integers(-20, 21, size=(N, 2, 2))
C = A @ B

X = np.hstack([A.reshape(N, 4), B.reshape(N, 4)]).astype(float)   # (50000, 8)
Y = C.reshape(N, 4).astype(float)                                  # (50000, 4)
print(X.shape, Y.shape)
print("Ejemplo:", A[0].tolist(), "x", B[0].tolist(), "=", C[0].tolist())

## 2. Partición del data set (sección 6.8.1)
60 % entrenamiento, 20 % validación, 20 % prueba. Los datos ya son aleatorios, así que basta cortar en orden.

In [ ]:
n_tr, n_va = int(0.6 * N), int(0.2 * N)
X_tr, Y_tr = X[:n_tr], Y[:n_tr]
X_va, Y_va = X[n_tr:n_tr+n_va], Y[n_tr:n_tr+n_va]
X_te, Y_te = X[n_tr+n_va:], Y[n_tr+n_va:]
print(len(X_tr), len(X_va), len(X_te))

## 3a. Red neuronal (MLP) 8 → 64 → 64 → 4
Normalizamos para que la red entrene bien: entradas ÷ 20 (quedan en [-1, 1]) y salidas ÷ 800 (el máximo posible de |cᵢⱼ| es 2·20·20 = 800). Al predecir se des-normaliza.

In [ ]:
XS, YS = 20.0, 800.0

mlp = MLPRegressor(hidden_layer_sizes=(64, 64), activation="relu",
                   solver="adam", learning_rate_init=1e-3,
                   max_iter=300, early_stopping=True, n_iter_no_change=15,
                   random_state=0)

t0 = time.perf_counter()
mlp.fit(X_tr / XS, Y_tr / YS)
t_train_mlp = time.perf_counter() - t0

pred_mlp = lambda x: mlp.predict(x / XS) * YS

print(f"Épocas: {mlp.n_iter_} | tiempo de entrenamiento: {t_train_mlp:.1f} s")
for nombre, Xs, Ys in [("validación", X_va, Y_va), ("prueba", X_te, Y_te)]:
    p = pred_mlp(Xs)
    exactos = np.mean(np.all(np.rint(p) == Ys, axis=1)) * 100
    print(f"{nombre}: error abs. medio = {mean_absolute_error(Ys, p):.3f} | matrices exactas = {exactos:.1f}%")

## 3b. Regresión polinomial de grado 2 (para comparar)
Cada cᵢⱼ = aᵢ₁·b₁ⱼ + aᵢ₂·b₂ⱼ, o sea una suma de productos de **dos** variables. Si le damos al modelo los términos de grado 2 (`PolynomialFeatures`), una regresión lineal normal puede encontrar los coeficientes exactos.

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)

t0 = time.perf_counter()
lin = LinearRegression().fit(poly.fit_transform(X_tr), Y_tr)
t_train_poly = time.perf_counter() - t0

pred_poly = lambda x: lin.predict(poly.transform(x))

p = pred_poly(X_te)
print(f"Tiempo de entrenamiento: {t_train_poly:.2f} s")
print(f"prueba: error abs. medio = {mean_absolute_error(Y_te, p):.2e} | "
      f"matrices exactas = {np.mean(np.all(np.rint(p) == Y_te, axis=1))*100:.1f}%")

## 4. Diez ejemplos nuevos: analítico vs. modelos

In [ ]:
A10 = rng.integers(-20, 21, size=(10, 2, 2))
B10 = rng.integers(-20, 21, size=(10, 2, 2))
X10 = np.hstack([A10.reshape(10, 4), B10.reshape(10, 4)]).astype(float)

real   = (A10 @ B10).reshape(10, 4)     # cálculo analítico
p_mlp  = pred_mlp(X10)
p_poly = pred_poly(X10)

for i in range(10):
    print(f"Ejemplo {i+1}: A={A10[i].tolist()}  B={B10[i].tolist()}")
    print(f"  Analítico : {real[i].tolist()}")
    print(f"  MLP       : {np.round(p_mlp[i], 2).tolist()}  (error máx = {np.abs(p_mlp[i]-real[i]).max():.2f})")
    print(f"  Polinomial: {np.round(p_poly[i], 2).tolist()}  (error máx = {np.abs(p_poly[i]-real[i]).max():.1e})\n")

## 5. Costo computacional
- **Analítico:** 8 multiplicaciones + 4 sumas = 12 operaciones por producto.
- **MLP:** por predicción hace 8·64 + 64·64 + 64·4 = 4.864 multiplicaciones-acumulaciones, y además hubo que entrenarla.

In [ ]:
def analitico(a, b):
    return [a[0][0]*b[0][0] + a[0][1]*b[1][0], a[0][0]*b[0][1] + a[0][1]*b[1][1],
            a[1][0]*b[0][0] + a[1][1]*b[1][0], a[1][0]*b[0][1] + a[1][1]*b[1][1]]

Al, Bl = A10.tolist(), B10.tolist()

rep = 20_000
t0 = time.perf_counter()
for _ in range(rep):
    for i in range(10):
        analitico(Al[i], Bl[i])
t_an = (time.perf_counter() - t0) / (rep * 10)

rep = 300
t0 = time.perf_counter()
for _ in range(rep):
    for i in range(10):
        mlp.predict(X10[i:i+1] / XS)
t_ml = (time.perf_counter() - t0) / (rep * 10)

macs_mlp   = 8*64 + 64*64 + 64*4
macs_train = 3 * macs_mlp * n_tr * mlp.n_iter_     # forward + backward ≈ 3 pasadas

print(f"Analítico: 12 operaciones | {t_an*1e6:.2f} µs por producto")
print(f"MLP      : {macs_mlp} MACs     | {t_ml*1e6:.1f} µs por producto")
print(f"Entrenar la MLP: ~{macs_train:.2e} operaciones, {t_train_mlp:.1f} s (una sola vez)")
print(f"MLP/analítico -> operaciones: ~{macs_mlp/12:.0f}x | tiempo: ~{t_ml/t_an:.0f}x")

## Conclusiones
- La **MLP** aproxima el producto pero con error (no es exacta); la **regresión polinomial** es exacta porque el problema es justo de grado 2.
- El **cálculo analítico es mucho más barato** (cientos de veces menos operaciones, sin entrenamiento) y exacto.
- ML se justifica cuando **no existe una fórmula conocida** (spam, traducción, reconocimiento de imágenes), no para multiplicar matrices.